In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the dataset
df = pd.read_csv('Ecommerce_Transactions_Clean.csv')

# 2. Filter for successful transactions to measure true customer retention
df = df[df['Payment_Status'] == 'Success'].copy()

# 3. Convert Purchase_Date to datetime format
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])

df.head()

,Transaction_ID,Customer_ID,Purchase_Date,Product_Category,Product_Name,Quantity,Unit_Price,Total_Amount,Payment_Status,Acquisition_Channel,Customer_Segment,City,State,Gender,Age_Group
0,T000001,C0001,2025-11-13,Electronics,Monitor,3,2398,7194,Success,Organic Search,New,Mumbai,Maharashtra,Male,45-54
1,T000002,C0002,2025-08-16,Books,Novel,1,999,999,Success,Google Ads,New,Kolkata,West Bengal,Male,25-34
2,T000003,C0002,2025-09-30,Accessories,Wallet,3,2999,8997,Success,Google Ads,New,Kolkata,West Bengal,Male,25-34
3,T000004,C0003,2025-07-04,Beauty,Perfume,3,299,897,Success,Instagram,New,Chennai,Tamil Nadu,Male,25-34
4,T000005,C0004,2025-09-13,Electronics,Headphones,1,7498,7498,Success,Facebook Ads,VIP,Delhi,Delhi,Male,45-54


In [5]:
# Convert purchase date to monthly period (YYYY-MM)
df['Tx_Month'] = df['Purchase_Date'].dt.to_period('M')

# Get the first purchase month for each customer
df['Cohort_Month'] = df.groupby('Customer_ID')['Tx_Month'].transform('min')

# Calculate month difference (0 = Month 0, 1 = Month 1, etc.)
years_diff = df['Tx_Month'].dt.year - df['Cohort_Month'].dt.year
months_diff = df['Tx_Month'].dt.month - df['Cohort_Month'].dt.month

df['Cohort_Index'] = years_diff * 12 + months_diff
df[['Customer_ID', 'Cohort_Month', 'Tx_Month', 'Cohort_Index']].head()

,Customer_ID,Cohort_Month,Tx_Month,Cohort_Index
0,C0001,2025-11,2025-11,0
1,C0002,2025-08,2025-08,0
2,C0002,2025-08,2025-09,1
3,C0003,2025-07,2025-07,0
4,C0004,2025-09,2025-09,0


In [6]:
# Pivot count of unique active customers
counts_matrix = df.groupby(['Cohort_Month', 'Cohort_Index'])['Customer_ID'].nunique().unstack()

print("--- ABSOLUTE ACTIVE USER COUNTS ---")
counts_matrix

--- ABSOLUTE ACTIVE USER COUNTS ---


Cohort_Index,0,1,2,3,4,5,6
Cohort_Month,,,,,,,
2025-01,60.0,23.0,25.0,10.0,2.0,NaN,NaN
2025-02,74.0,36.0,18.0,13.0,2.0,1.0,NaN
2025-03,72.0,34.0,27.0,15.0,7.0,3.0,NaN
2025-04,69.0,29.0,26.0,7.0,1.0,1.0,NaN
2025-05,53.0,23.0,15.0,8.0,3.0,NaN,NaN
2025-06,57.0,33.0,23.0,16.0,10.0,4.0,1.0
2025-07,54.0,26.0,11.0,7.0,6.0,4.0,NaN
2025-08,69.0,28.0,20.0,9.0,3.0,1.0,NaN
2025-09,78.0,33.0,30.0,15.0,4.0,4.0,NaN
